<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

// Интерфейсы для множественного наследования
public interface ITransactionLogger
{
    void LogTransaction(string transactionDetails);
    string GetTransactionHistory();
}

public interface ISecurityValidator
{
    bool ValidateSecurity();
    void SetSecurityLevel(int level);
}

public interface ICurrencyConverter
{
    decimal ConvertAmount(decimal amount, string targetCurrency);
    string DefaultCurrency { get; set; }
}

// Базовый абстрактный класс с новыми атрибутами и методами
public abstract class PaymentMethod : ITransactionLogger, ISecurityValidator
{
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }
    
    // Новые атрибуты
    public DateTime CreatedDate { get; set; }
    public bool IsActive { get; set; }
    public string Currency { get; set; }
    public int SecurityLevel { get; set; }
    protected List<string> TransactionHistory { get; set; }
    
    protected PaymentMethod() 
    {
        CreatedDate = DateTime.Now;
        IsActive = true;
        Currency = "RUB";
        SecurityLevel = 1;
        TransactionHistory = new List<string>();
    }
    
    protected PaymentMethod(int id, string name, decimal minAmount) : this()
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
    }

    public virtual void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Произведена транзакция через {MethodName} в размере: {amount}");
        LogTransaction($"Транзакция через {MethodName} в размере: {amount}");
    }

    public virtual bool CheckMinimumAmount(decimal amount)
    {
        if (amount >= MinAmount)
        {
            Console.WriteLine("Средств: Достаточно");
            return true;
        }
        else
        {
            Console.WriteLine("Средств: Недостаточно");
            return false;
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId} \nСпособ оплаты: {MethodName} \nМинимальная сумма: {MinAmount}");
    }

    public void ProcessPaymentWithCheck(decimal amount)
    {
        if (CheckMinimumAmount(amount) && ValidateSecurity())
        {
            ProcessPayment(amount);
        }
        else
        {
            Console.WriteLine("Транзакция отменена");
        }
    }

    // Новые методы базового класса
    public virtual void Deactivate()
    {
        IsActive = false;
        Console.WriteLine($"Способ оплаты {MethodName} деактивирован");
    }

    public virtual void Activate()
    {
        IsActive = true;
        Console.WriteLine($"Способ оплаты {MethodName} активирован");
    }

    // Реализация интерфейса ITransactionLogger
    public void LogTransaction(string transactionDetails)
    {
        string logEntry = $"{DateTime.Now:dd.MM.yyyy HH:mm:ss} - {transactionDetails}";
        TransactionHistory.Add(logEntry);
    }

    public string GetTransactionHistory()
    {
        if (TransactionHistory.Count == 0)
            return "История транзакций пуста";
        
        return string.Join("\n", TransactionHistory);
    }

    // Реализация интерфейса ISecurityValidator
    public virtual bool ValidateSecurity()
    {
        if (SecurityLevel >= 1 && IsActive)
        {
            return true;
        }
        else
        {
            Console.WriteLine("Проверка безопасности: НЕ ПРОЙДЕНА");
            return false;
        }
    }

    public virtual void SetSecurityLevel(int level)
    {
        SecurityLevel = level;
    }
}

class OnlinePayment : PaymentMethod, ICurrencyConverter
{
    public string PaymentUrl { get; set; }
    public string ApiKey { get; set; }
    public bool IsSecureConnection { get; set; }
    public string DefaultCurrency { get; set; }
    
    public OnlinePayment() : base() 
    {
        DefaultCurrency = "RUB";
    }
    
    public OnlinePayment(int id, string name, decimal minAmount, string paymentUrl) 
        : base(id, name, minAmount)
    {
        PaymentUrl = paymentUrl;
        IsSecureConnection = true;
        DefaultCurrency = "RUB";
    }
    
    public OnlinePayment(string paymentUrl) : this(1, "Онлайн", 100m, paymentUrl) { }
    
    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"по URL: {PaymentUrl}");
    }
    
    // Реализация интерфейса ICurrencyConverter
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal convertedAmount = amount * 0.85m;
        Console.WriteLine($"Конвертация: {amount} {Currency} → {convertedAmount} {targetCurrency}");
        return convertedAmount;
    }
}

class BankTransfer : PaymentMethod
{
    public decimal BankFee { get; set; }
    public string BankAccount { get; set; }
    public string RecipientName { get; set; }
    
    public BankTransfer() : base() { }
    
    public BankTransfer(int id, string name, decimal minAmount, decimal bankFee) 
        : base(id, name, minAmount)
    {
        BankFee = bankFee;
    }
    
    public BankTransfer(decimal bankFee) : this(2, "Банковский перевод", 1000m, bankFee) { }
    
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal totalAmount = amount + BankFee;
        if (totalAmount >= MinAmount)
        {
            Console.WriteLine($"Средств: Достаточно (включая комиссию банка: {BankFee})");
            return true;
        }
        else
        {
            Console.WriteLine($"Средств: Недостаточно (требуется минимум: {MinAmount}, включая комиссию)");
            return false;
        }
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Комиссия банка: {BankFee}");
    }
}

class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public string OperatorName { get; set; }
    
    public CashPayment() : base() { }
    
    public CashPayment(int id, string name, decimal minAmount, string cashPickupPoint) 
        : base(id, name, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
    }
    
    public CashPayment(string cashPickupPoint) : this(3, "Наличные", 150m, cashPickupPoint) { }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место выдачи наличных: {CashPickupPoint}");
    }
}

// Новый класс для демонстрации сложного наследования
class CreditCardPayment : PaymentMethod, ICurrencyConverter
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }
    public string DefaultCurrency { get; set; }
    
    public CreditCardPayment() : base() 
    {
        DefaultCurrency = "RUB";
    }
    
    public CreditCardPayment(int id, string name, decimal minAmount, string cardNumber, string cardHolder) 
        : base(id, name, minAmount)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
        DefaultCurrency = "RUB";
    }
    
    public CreditCardPayment(string cardNumber, string cardHolder) : this(4, "Кредитная карта", 50m, cardNumber, cardHolder) { }
    
    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"по карте: ****{CardNumber.Substring(CardNumber.Length - 4)}");
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Держатель карты: {CardHolder}");
        Console.WriteLine($"Номер карты: ****{CardNumber.Substring(CardNumber.Length - 4)}");
    }
    
    // Реализация интерфейса ICurrencyConverter
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal convertedAmount = amount * 0.88m;
        Console.WriteLine($"Конвертация по карте: {amount} {Currency} → {convertedAmount} {targetCurrency}");
        return convertedAmount;
    }
}

class PaymentProcessor
{
    private List<PaymentMethod> _availablePaymentMethods;
    
    public PaymentProcessor()
    {
        _availablePaymentMethods = new List<PaymentMethod>();
    }
    
    public void AddPaymentMethod(PaymentMethod paymentMethod)
    {
        _availablePaymentMethods.Add(paymentMethod);
        Console.WriteLine($"Добавлен способ оплаты: {paymentMethod.MethodName}");
    }
    
    public void ProcessAllPayments(decimal amount)
    {
        Console.WriteLine($"\n=== Обработка всех доступных платежей на сумму: {amount} ===");
        
        foreach (var paymentMethod in _availablePaymentMethods)
        {
            Console.WriteLine($"\n--- Проверка: {paymentMethod.MethodName} ---");
            paymentMethod.ProcessPaymentWithCheck(amount);
        }
    }
    
    public void ShowAllPaymentDetails()
    {
        Console.WriteLine("\n=== Детали всех способов оплаты ===");
        foreach (var paymentMethod in _availablePaymentMethods)
        {
            Console.WriteLine("------------------------------");
            paymentMethod.GetPaymentDetails();
        }
    }
    
    public PaymentMethod FindPaymentMethodById(int id)
    {
        return _availablePaymentMethods.Find(pm => pm.PaymentMethodId == id);
    }
}

PaymentProcessor processor = new PaymentProcessor();

OnlinePayment onlinePayment = new OnlinePayment("14857291098459839")
{
    MinAmount = 100
};

BankTransfer bankTransfer = new BankTransfer(120)
{
    MinAmount = 1000
};

CashPayment cashPayment = new CashPayment("Сбербанк")
{
    MinAmount = 150
};

// Новый способ оплаты с множественным наследованием
CreditCardPayment creditCard = new CreditCardPayment("4111111111111111", "Иванов И.И.")
{
    MinAmount = 50
};

processor.AddPaymentMethod(onlinePayment);
processor.AddPaymentMethod(bankTransfer);
processor.AddPaymentMethod(cashPayment);
processor.AddPaymentMethod(creditCard);

decimal paymentAmount = 520;

Console.WriteLine("\n" + new string('=', 50));
processor.ShowAllPaymentDetails();

Console.WriteLine("\n" + new string('=', 50));
processor.ProcessAllPayments(paymentAmount);

Console.WriteLine("\n" + new string('=', 50));
Console.WriteLine("Поиск способа оплаты по ID:");

PaymentMethod foundPayment = processor.FindPaymentMethodById(2);
if (foundPayment != null)
{
    Console.WriteLine("Найден способ оплаты:");
    foundPayment.GetPaymentDetails();
}

Console.WriteLine("\n" + new string('=', 50));
Console.WriteLine("Индивидуальная обработка:");

onlinePayment.GetPaymentDetails();
onlinePayment.ProcessPaymentWithCheck(paymentAmount);

Console.WriteLine("\n------------------------------");

bankTransfer.GetPaymentDetails();
bankTransfer.ProcessPaymentWithCheck(paymentAmount);

Console.WriteLine("\n------------------------------");

cashPayment.GetPaymentDetails();
cashPayment.ProcessPaymentWithCheck(paymentAmount);

Console.WriteLine("\n------------------------------");

creditCard.GetPaymentDetails();
creditCard.ProcessPaymentWithCheck(paymentAmount);

// Демонстрация новых возможностей
Console.WriteLine("\n" + new string('=', 50));
Console.WriteLine("Конвертация:");

Console.WriteLine("\n--- Конвертация валют ---");
onlinePayment.ConvertAmount(1000, "USD");
creditCard.ConvertAmount(1000, "EUR");

Console.WriteLine("\n--- История транзакций ---");
Console.WriteLine("История онлайн платежей:");
Console.WriteLine(onlinePayment.GetTransactionHistory());

Добавлен способ оплаты: Онлайн
Добавлен способ оплаты: Банковский перевод
Добавлен способ оплаты: Наличные
Добавлен способ оплаты: Кредитная карта


=== Детали всех способов оплаты ===
------------------------------
ID: 1 
Способ оплаты: Онлайн 
Минимальная сумма: 100
------------------------------
ID: 2 
Способ оплаты: Банковский перевод 
Минимальная сумма: 1000
Комиссия банка: 120
------------------------------
ID: 3 
Способ оплаты: Наличные 
Минимальная сумма: 150
Место выдачи наличных: Сбербанк
------------------------------
ID: 4 
Способ оплаты: Кредитная карта 
Минимальная сумма: 50
Держатель карты: Иванов И.И.
Номер карты: ****1111


=== Обработка всех доступных платежей на сумму: 520 ===

--- Проверка: Онлайн ---
Средств: Достаточно
Произведена транзакция через Онлайн в размере: 520
по URL: 14857291098459839

--- Проверка: Банковский перевод ---
Средств: Недостаточно (требуется минимум: 1000, включая комиссию)
Транзакция отменена

--- Проверка: Наличные ---
Средств: Достаточно
